In [0]:
%%capture --no-stderr
%pip install --quiet -U databricks-langchain langchain_core langgraph langgraph-prebuilt
dbutils.library.restartPython()

In [0]:
from langgraph.graph import START, END, StateGraph, MessagesState
from typing import TypedDict
from langgraph.prebuilt import ToolNode, tools_condition

In [0]:
from databricks_langchain import ChatDatabricks
llm = ChatDatabricks(model='agents-demo-gpt4o')

In [0]:
def multiply(x: int, y: int) -> int:
    """Multiply numbers x and y
    
    Args:
      x: first number
      y: second number
    """
    return x * y

llm_with_tool = llm.bind_tools([multiply])

In [0]:
class State(MessagesState):
  pass

In [0]:
from langchain_core.messages import HumanMessage, AIMessage
def tool_calling_llm(state: State) -> State:
    return {'messages': [llm_with_tool.invoke(state['messages'])]}

In [0]:
graph = StateGraph(State)

graph.add_node('tool_calling_llm', tool_calling_llm)
graph.add_node('tools', ToolNode([multiply]))

graph.add_edge(START, 'tool_calling_llm')
graph.add_conditional_edges('tool_calling_llm', tools_condition)
graph.add_edge('tools', END)


app = graph.compile()
app

In [0]:
messages = [HumanMessage(content="Hello, what is 2 multiplied by 2?")]
messages = app.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()